In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

train = pd.read_csv(BASE_DIR / "data" / "train.csv")
test = pd.read_csv(BASE_DIR / "data" / "test.csv")

print("Размер train:", train.shape)
print("Размер test:", test.shape)
train.head()

Обучающая выборка содержит 891 пассажира и 12 колонок. В тестовой выборке 418 пассажиров и нет целевой колонки `Survived`.

In [ ]:
train.describe(include="all").T

## Пропуски

In [ ]:
missing = pd.DataFrame({
    "train": train.isna().sum(),
    "test": test.isna().sum()
}).fillna(0).astype(int)

missing[missing.sum(axis=1) > 0]

В `train` пропущены 177 значений возраста, 687 значений каюты и 2 значения порта посадки. В `test` также отсутствуют 86 возрастов, 327 кают и одна стоимость билета.

Возраст и стоимость билета — числовые признаки. В pipeline XGBoost они заполняются медианой, а CatBoost умеет сам работать с числовыми пропусками. Для `Embarked` в XGBoost используется самое частое значение, а в CatBoost — отдельная категория `Unknown`. `Cabin` не входит в модель: в этой колонке отсутствует слишком большая часть данных.

## Распределение целевой переменной

In [ ]:
survival_percent = train["Survived"].value_counts(normalize=True).sort_index() * 100
print(survival_percent.rename(index={0: "Не выжил", 1: "Выжил"}))

plt.figure(figsize=(6, 4))
sns.countplot(data=train, x="Survived", color="steelblue")
plt.xticks([0, 1], ["Не выжил", "Выжил"])
plt.title("Распределение выживаемости")
plt.xlabel("Результат")
plt.ylabel("Количество пассажиров")
plt.show()

Выжили около 38.4% пассажиров, а 61.6% не выжили. Классы распределены не поровну, поэтому для оценки используется `StratifiedKFold`: в каждом фолде сохраняется примерно одинаковая доля классов.

## Связь пола и класса билета с выживаемостью

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(
    data=train, x="Sex", y="Survived",
    errorbar=None, color="steelblue", ax=axes[0]
)
axes[0].set_title("Доля выживших по полу")
axes[0].set_xlabel("Пол")
axes[0].set_ylabel("Доля выживших")

sns.barplot(
    data=train, x="Pclass", y="Survived",
    errorbar=None, color="darkorange", ax=axes[1]
)
axes[1].set_title("Доля выживших по классу билета")
axes[1].set_xlabel("Класс билета")
axes[1].set_ylabel("Доля выживших")

plt.tight_layout()
plt.show()

print("По полу:")
print((train.groupby("Sex")["Survived"].mean() * 100).round(2))
print("\nПо классу:")
print((train.groupby("Pclass")["Survived"].mean() * 100).round(2))

Пол и класс билета сильно связаны с результатом. Среди женщин выжили около 74.2%, среди мужчин — 18.9%. В первом классе выжили около 63.0% пассажиров, а в третьем — только 24.2%. Поэтому `Sex` и `Pclass` включены в признаки модели.

## Возраст и выживаемость

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=train, x="Age", hue="Survived",
    bins=30, kde=True, element="step", common_norm=False
)
plt.title("Распределение возраста по выживаемости")
plt.xlabel("Возраст")
plt.ylabel("Количество пассажиров")
plt.show()

Распределения возраста выживших и погибших заметно пересекаются, поэтому один возраст не позволяет хорошо разделить классы. При этом среди маленьких детей видна более высокая доля выживших. Возраст оставлен в модели как дополнительный числовой признак.

## Итоговые выводы

- Самые заметные зависимости выживаемости связаны с полом и классом билета.
- Для числовых пропусков в XGBoost используется медиана, так как она устойчива к большим значениям стоимости билета. CatBoost обрабатывает числовые пропуски самостоятельно.
- Пропуски `Embarked` в XGBoost заполняются самым частым значением, а в CatBoost значением `Unknown`.
- `Cabin` не используется из-за очень большого количества пропусков.
- Категориальные признаки кодируются внутри pipeline, а CatBoost получает их как категории.
- Из-за различия в долях классов для CV используется стратифицированное разбиение.